# RF-DETR-Seg - Instance Segmentation on Geospatial Imagery

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/rfdetr_segmentation.ipynb)

This notebook demonstrates GeoAI's RF-DETR-Seg integration for instance segmentation on GeoTIFF imagery.

`rfdetr_segment()` runs tiled inference with RF-DETR-Seg, preserves the predicted instance masks, and returns a georeferenced GeoDataFrame with mask polygons. You can also call `rfdetr_detect()` with a `seg-*` model variant; mask geometry is enabled automatically for segmentation variants.

In [ ]:
# %pip install -U "geoai-py[rfdetr]"

## Import Libraries

In [ ]:
import geoai

## List RF-DETR-Seg Models

GeoAI registers the RF-DETR detection and segmentation variants. The segmentation variants are named with the `seg-` prefix.

In [ ]:
from geoai import RFDETR_MODELS, list_rfdetr_models

for name, desc in list_rfdetr_models().items():
    if name.startswith("seg-"):
        resolution = RFDETR_MODELS[name]["resolution"]
        print(f"{name:12s} [{resolution:4d}px] {desc}")

## Download a Sample GeoTIFF

This example uses a public GeoTIFF from the GeoAI sample data collection. The default RF-DETR-Seg weights are trained on general COCO objects, so for domain-specific classes such as buildings, ships, tree crowns, or wetlands, use a fine-tuned RF-DETR-Seg checkpoint with `pretrain_weights=`.

In [ ]:
raster_url = (
    "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/parking_spots.tif"
)
raster_path = geoai.download_file(raster_url)
raster_path

## View the Input Image

In [ ]:
geoai.view_raster(raster_url)

## Run RF-DETR-Seg

`rfdetr_segment()` returns georeferenced mask polygons. The `area_pixels` column records the pixel area of each predicted mask before vectorization.

In [ ]:
gdf = geoai.rfdetr_segment(
    raster_path,
    output_path="rfdetr_segmentation.geojson",
    model_variant="seg-medium",
    confidence_threshold=0.35,
    nms_threshold=0.3,
    batch_size=2,
)

print(f"Segmented {len(gdf)} objects")
gdf.head()

## Inspect the Segmentation Results

In [ ]:
print(gdf.geom_type.value_counts().to_string())
print(gdf["class_name"].value_counts().head(10).to_string())
print(f"Mean mask area: {gdf['area_pixels'].mean():.1f} pixels")

## Visualize Mask Polygons

In [ ]:
geoai.view_vector_interactive(gdf, column="class_name", tiles=raster_url)

## Compare with Bounding Boxes

For segmentation variants, `rfdetr_detect()` uses mask polygons by default. Set `use_mask_geometry=False` when you want bounding box rectangles from the same RF-DETR-Seg predictions.

In [ ]:
boxes = geoai.rfdetr_detect(
    raster_path,
    model_variant="seg-medium",
    confidence_threshold=0.35,
    nms_threshold=0.3,
    batch_size=2,
    use_mask_geometry=False,
)

print(f"Bounding boxes: {len(boxes)}")
geoai.view_vector_interactive(boxes, column="class_name", tiles=raster_url)

## Use Custom Weights

Pass a fine-tuned RF-DETR-Seg checkpoint through `pretrain_weights` to segment custom geospatial classes.

```python
custom_gdf = geoai.rfdetr_segment(
    raster_path,
    model_variant="seg-medium",
    pretrain_weights="path/to/checkpoint_best_total.pth",
    class_names=["building", "tree", "vehicle"],
    confidence_threshold=0.3,
)
```

## Save Results

The earlier `output_path` argument already saved the segmentation output to GeoJSON. You can also write the GeoDataFrame to any vector format supported by GeoPandas.

In [ ]:
gdf.to_file("rfdetr_segmentation.gpkg", driver="GPKG")
print("Saved RF-DETR-Seg mask polygons to rfdetr_segmentation.gpkg")